In [1]:
import torch
import Mat1.config as c
from utils import trainer as t

In [2]:
config = c.get_config(0, run_id="8", hidden_layers=[1024]*1, mode="rfa")

[LinearRFA(), LinearRFA()]


In [3]:
t.train_network(config, num_epochs=100, checkpoint_interval=50)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Initializing Wandb:


wandb: Currently logged in as: g-aravindadithya (Trainers100) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W0408 20:48:01.781000 33772 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0408 20:48:01.783000 33772 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0408 20:48:01.785000 33772 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0408 20:48:01.786000 33772 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 

[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/venv/main/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
EPOCH:  1
Computing GOP for sample 0 out of 10
Computing GOP for sample 1 out of 10
Computing GOP for sample 2 out of 10
Computing GOP for sample 3 out of 10
Computing GOP for sample 4 out of 10
Computing GOP for sample 5 out of 10
Computing GOP for sample 6 out of 10
Computing GOP for sample 7 out of 10
Computing GOP for sample 8 out of 10
Computing GOP for sample 9 out of 10
Computing GOP for sample 10 out of 10
Time:  3.2025442123413086
Generating Visuals...
Visuals Logging Completed.
EPOCH:  2
Computing GOP for sample 0 out of 10
Computing GOP for sample 1 out of 10
Computing GOP for sample 2 out of 10
Computing GOP for sample 3 out of 10
Computing GOP for sample 4 out of 10
Computing GOP for sample 5 out of 10
Computing GOP for sample 6 out of 10
Computing GOP for sample 7 out of 10
Computing GOP for sample 8 out of 10
Computing GOP for sample 9 out of 

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7cbb30aa9d90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7cbb30aa97f0, execution_count=3 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7cbb30aa97c0, raw_cell="t.train_network(config, num_epochs=100, checkpoint.." transformed_cell="t.train_network(config, num_epochs=100, checkpoint.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://dev-container%2B7b227265706f7369746f727950617468223a2268747470733a2f2f6769746875622e636f6d2f61726176696e64616469746879612f4d61747269672e6769742f747265652f6d6173746572222c22766f6c756d654e616d65223a224d61747269672d6d61737465722d62353132393936373030653663336536623532643661366636306537636564613937326161383937663434346361666233353766633066363632323230663739222c22666f6c646572223a224d6174726967222c2273657474696e6773223a7b22636f6e74657874223a226465736b746f702d6c696e7578227d2c2

ConnectionResetError: Connection lost

In [7]:
print(config['net'])
net = config['net']


Net(
  (features): Sequential(
    (0): Linear(in_features=784, out_features=1024, bias=False)
  )
  (classifier): Linear(in_features=1024, out_features=784, bias=False)
)


In [1]:
''' This module does the following
1. Scan the network for conv layers
2. For each FC layer compute W^TW of eq 3
3. For each FC layer compute the AGOP(AJOP in case of multiple outputs)
4. For each conv layer print the pearson correlation between 2 and 3
'''

import torch
import torch.nn as nn
import random
import numpy as np
#from functorch import jacrev, vmap
from torch.func import jacrev
from torch.nn.functional import pad
#import dataset
from numpy.linalg import eig
from copy import deepcopy
from torch.linalg import norm, svd
from torchvision import models
from torch.linalg import norm, eig

from utils.linear_rfa import LinearRFA


def get_jacobian(net, data, c_idx=0, chunk=100):
    with torch.no_grad():
        def single_net(x):
            # x is (s)
            return net(x.unsqueeze(0))[:,c_idx*chunk:(c_idx+1)*chunk].squeeze(0)
        # Parallelize across the images.
        return torch.vmap(jacrev(single_net))(data) #(n,chunk,s)

def min_max(M):
    return (M - M.min()) / (M.max() - M.min())

def sqrt(G):
    U, s, Vt = svd(G)
    s = torch.pow(s, 1./2)
    G = U @ torch.diag(s) @ Vt
    return G


def correlation(M1, M2):
    M1 -= M1.mean()
    M2 -= M2.mean()

    norm1 = norm(M1.flatten())
    norm2 = norm(M2.flatten())

    return torch.sum(M1.cuda() * M2.cuda()) / (norm1 * norm2)



def egop(model, z, c=10, chunk_idxs=1):
    ajop = 0
    #Chunking is done to compute jacobian as chunks. This saves memory
    chunk = c // chunk_idxs
    for i in range(chunk_idxs):
        grads = get_jacobian(model, z, c_idx=i, chunk=chunk) #(n,chunk,s)
        grads_t = grads.transpose(1, 2) 
        ajop_matmul= torch.matmul(grads_t, grads) #(n,s,s)
        #Clarify: mean and sum are making no difference here. Check if trainloader has grouped images
        ajop += torch.mean(ajop_matmul, dim=0) #(s,s)
    return ajop


def get_grads(net, patchnet, trainloader, max_batch, classes, chunk_idx,
              kernel=(3,3), padding=(1,1),
              stride=(1,1), layer_idx=0):
    net.eval()
    net.cuda()
    patchnet.eval()
    patchnet.cuda()
    M = 0
    #M.cuda()
    
    # Num images for taking AGOP (Can be small for early layers)
    MAX_NUM_IMGS = max_batch

    for idx, batch in enumerate(trainloader):
        print("Computing GOP for sample " + str(idx) + \
              " out of " + str(MAX_NUM_IMGS))
        imgs, _ = batch
        #imgs=imgs[:]
        with torch.no_grad():
            imgs = imgs.cuda()        
            # Run the first half of the network wrt to the current layer 
            ip = net.features[:layer_idx](imgs).cpu() #(n,s)
            
        #print(patches.shape)
        M += egop(patchnet,ip.cuda(), classes, chunk_idx).cuda()
        del imgs
        torch.cuda.empty_cache()
        if idx >= MAX_NUM_IMGS:
            break
    net.cpu()
    patchnet.cpu()
    return M
    

def load_nn(net, init_net, layer_idx=0):
   
    count = 0
    
    # Get the layer_idx+1 th conv layer
    #TODO: Add functionality to access classifier layers too.
    for idx, m in enumerate(net.features):
        if isinstance(m, (nn.Linear, LinearRFA)):
            count += 1
        if count-1 == layer_idx:
            l_idx = idx
            break
    
    patchnet = deepcopy(net)
    
    # Truncate all layers before l_idx.
    patchnet.features = net.features[l_idx:]
    
    M = net.features[l_idx].weight.data
    # Compute WW which is (s,s) matrix
    M =torch.matmul(M.T, M)
    M0 = init_net.features[l_idx].weight.data
    # Compute W0tW0 which is (s,s) matrix
    M0 =torch.matmul(M0.T, M0)
    return net, patchnet, M, M0, l_idx

def verify_NFA(net, init_net, trainloader, layer_idx=0, max_batch=10, classes=10, chunk_idx=1):

    # Use copies so diagnostic device moves do not change the training model.
    net_copy = deepcopy(net)
    init_net_copy = deepcopy(init_net)

    net_copy, patchnet, M, M0, l_idx = load_nn(net_copy, init_net_copy, layer_idx=layer_idx)

    #i_val = correlation(M0.cuda(), M.cuda())
    # print("Correlation between Initial and Trained CNFM: ", i_val)

    G = get_grads(net_copy, patchnet, trainloader,  max_batch, classes, chunk_idx,
                  layer_idx=l_idx)
    # print("Shape of grad matrix",G.shape)
    G = sqrt(G.cuda())
    r_val = correlation(M.cuda(), G.cuda())
    print("Correlation between Trained CNFM and AGOP: ", r_val)
    # print("Final: ", i_val, r_val)
    return {
        'agop_correlation_uncentered': r_val.item()
    }


In [ ]:
verify_NFA(net, net, config['trainloader'], layer_idx=0, max_batch=10, classes=784, chunk_idx=8)